# PART-A

In [993]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import time

In [994]:
# Load dataset
df = pd.read_csv("garments_worker_productivity.csv")

display(df.head())
print("Shape:", df.shape)
print(df.columns.tolist())
df.info()

# Missing values
print(df.isnull().sum())
display(df.describe().T)

# categorical columns
print("\nCategorical Columns:")
categorical_columns = df.select_dtypes(include=["object"]).columns.tolist()
print(categorical_columns)

for col in categorical_columns:
    print(f"\n{col}")
    print(df[col].unique())

print("\nTarget-related columns:")
print("actual_productivity:")
display(df["actual_productivity"].describe())

print("\ntargeted_productivity:")
display(df["targeted_productivity"].describe())

,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


Shape: (1197, 15)
['date', 'quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers', 'actual_productivity']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   object 
 1   quarter                1197 non-null   object 
 2   department             1197 non-null   object 
 3   day                    1197 non-null   object 
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null   float6

,count,mean,std,min,25%,50%,75%,max
team,1197.0,6.426901,3.463963,1.000000,3.000000,6.000000,9.000000,12.000000
targeted_productivity,1197.0,0.729632,0.097891,0.070000,0.700000,0.750000,0.800000,0.800000
smv,1197.0,15.062172,10.943219,2.900000,3.940000,15.260000,24.260000,54.560000
wip,691.0,1190.465991,1837.455001,7.000000,774.500000,1039.000000,1252.500000,23122.000000
over_time,1197.0,4567.460317,3348.823563,0.000000,1440.000000,3960.000000,6960.000000,25920.000000
incentive,1197.0,38.210526,160.182643,0.000000,0.000000,0.000000,50.000000,3600.000000
idle_time,1197.0,0.730159,12.709757,0.000000,0.000000,0.000000,0.000000,300.000000
idle_men,1197.0,0.369256,3.268987,0.000000,0.000000,0.000000,0.000000,45.000000
no_of_style_change,1197.0,0.150376,0.427848,0.000000,0.000000,0.000000,0.000000,2.000000
no_of_workers,1197.0,34.609858,22.197687,2.000000,9.000000,34.000000,57.000000,89.000000



Categorical Columns:
['date', 'quarter', 'department', 'day']

date
['1/1/2015' '1/3/2015' '1/4/2015' '1/5/2015' '1/6/2015' '1/7/2015'
 '1/8/2015' '1/10/2015' '1/11/2015' '1/12/2015' '1/13/2015' '1/14/2015'
 '1/15/2015' '1/17/2015' '1/18/2015' '1/19/2015' '1/20/2015' '1/21/2015'
 '1/22/2015' '1/24/2015' '1/25/2015' '1/26/2015' '1/27/2015' '1/28/2015'
 '1/29/2015' '1/31/2015' '2/1/2015' '2/2/2015' '2/3/2015' '2/4/2015'
 '2/5/2015' '2/7/2015' '2/8/2015' '2/9/2015' '2/10/2015' '2/11/2015'
 '2/12/2015' '2/14/2015' '2/15/2015' '2/16/2015' '2/17/2015' '2/18/2015'
 '2/19/2015' '2/22/2015' '2/23/2015' '2/24/2015' '2/25/2015' '2/26/2015'
 '2/28/2015' '3/1/2015' '3/2/2015' '3/3/2015' '3/4/2015' '3/5/2015'
 '3/7/2015' '3/8/2015' '3/9/2015' '3/10/2015' '3/11/2015']

quarter
['Quarter1' 'Quarter2' 'Quarter3' 'Quarter4' 'Quarter5']

department
['sweing' 'finishing ' 'finishing']

day
['Thursday' 'Saturday' 'Sunday' 'Monday' 'Tuesday' 'Wednesday']

Target-related columns:
actual_productivity:


count    1197.000000
mean        0.735091
std         0.174488
min         0.233705
25%         0.650307
50%         0.773333
75%         0.850253
max         1.120437
Name: actual_productivity, dtype: float64


targeted_productivity:


count    1197.000000
mean        0.729632
std         0.097891
min         0.070000
25%         0.700000
50%         0.750000
75%         0.800000
max         0.800000
Name: targeted_productivity, dtype: float64

In [995]:
# Create classification target
df["MeetsTarget"] = (
    df["actual_productivity"] >= df["targeted_productivity"]
).astype(int)

print(df["MeetsTarget"].value_counts())

print("\nClass proportions:")
print(df["MeetsTarget"].value_counts(normalize=True))

# Regression target and features
y_reg = df["actual_productivity"]
X_reg = df.drop(columns=["actual_productivity", "MeetsTarget"] ) # features
print("\nRegression feature shape:", X_reg.shape)
print("Regression target shape:", y_reg.shape)

# Classification target and features
y_cls = df["MeetsTarget"]
X_cls = df.drop(columns=["actual_productivity", "MeetsTarget"] )  # features
print("Classification feature shape:", X_cls.shape)
print("Classification target shape:", y_cls.shape)

MeetsTarget
1    875
0    322
Name: count, dtype: int64

Class proportions:
MeetsTarget
1    0.730994
0    0.269006
Name: proportion, dtype: float64

Regression feature shape: (1197, 14)
Regression target shape: (1197,)
Classification feature shape: (1197, 14)
Classification target shape: (1197,)


In [996]:
numeric_features = X_reg.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_reg.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features: \n", numeric_features)
print("\nCategorical features: \n", categorical_features)

Numerical features: 
 ['team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']

Categorical features: 
 ['date', 'quarter', 'department', 'day']


In [997]:
# Create row indices
indices = np.arange(len(df))

# Stratified train-test split
train_indices, test_indices = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    stratify=df["MeetsTarget"]
)

print("Training samples:", len(train_indices))
print("Testing samples:", len(test_indices))

# Check class proportions
print("\nOverall class distribution:")
print(df["MeetsTarget"].value_counts(normalize=True))

print("\nTraining class distribution:")
print(df["MeetsTarget"].iloc[train_indices].value_counts(normalize=True))

print("\nTesting class distribution:")
print(df["MeetsTarget"].iloc[test_indices].value_counts(normalize=True))

# Regression split
X_reg_train = X_reg.iloc[train_indices].copy()
X_reg_test = X_reg.iloc[test_indices].copy()

y_reg_train = y_reg.iloc[train_indices].copy()
y_reg_test = y_reg.iloc[test_indices].copy()

# Classification split
X_cls_train = X_cls.iloc[train_indices].copy()
X_cls_test = X_cls.iloc[test_indices].copy()

y_cls_train = y_cls.iloc[train_indices].copy()
y_cls_test = y_cls.iloc[test_indices].copy()

Training samples: 957
Testing samples: 240

Overall class distribution:
MeetsTarget
1    0.730994
0    0.269006
Name: proportion, dtype: float64

Training class distribution:
MeetsTarget
1    0.731452
0    0.268548
Name: proportion, dtype: float64

Testing class distribution:
MeetsTarget
1    0.729167
0    0.270833
Name: proportion, dtype: float64


In [998]:
# numeric pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# categorical pipeline
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

In [999]:
# combine numeric and catogoric pipeline for regression
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_reg_train_processed = preprocessor.fit_transform(X_reg_train)
X_reg_test_processed = preprocessor.transform(X_reg_test)

# combine numeric and catogoric pipeline for classification
preprocessor_cls = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_cls_train_processed = preprocessor_cls.fit_transform(X_cls_train)
X_cls_test_processed = preprocessor_cls.transform(X_cls_test)

In [1000]:
# applying linear regression
linear_model = LinearRegression()

start_train = time.perf_counter()
linear_model.fit(X_reg_train_processed, y_reg_train)
end_train = time.perf_counter()
linear_train_time = end_train - start_train

print(f"Training time: {linear_train_time:.6f} seconds")

# make predictions
start_predict = time.perf_counter()
y_reg_pred = linear_model.predict(X_reg_test_processed)
end_predict = time.perf_counter()
linear_prediction_time = end_predict - start_predict

print(f"Prediction time: {linear_prediction_time:.6f} seconds")

Training time: 0.013217 seconds
Prediction time: 0.000355 seconds


In [1001]:
# calculate MAE
linear_mae = mean_absolute_error(y_reg_test, y_reg_pred)
print("MAE: ", linear_mae)

# calculate RMSE
linear_rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
print("RMSE: ", linear_rmse)

# calculate R^2
linear_r2 = r2_score(y_reg_test, y_reg_pred)
print("R²: ", linear_r2)

# display result
linear_results = pd.DataFrame({
    "Model": ["Scikit-learn Linear Regression"],
    "MAE": [linear_mae],
    "RMSE": [linear_rmse],
    "R2": [linear_r2],
    "Training Time (s)": [linear_train_time],
    "Prediction Time (s)": [linear_prediction_time]
})
display(linear_results)

MAE:  0.1089679033925567
RMSE:  0.14597578427035596
R²:  0.26621914603589225


,Model,MAE,RMSE,R2,Training Time (s),Prediction Time (s)
0,Scikit-learn Linear Regression,0.108968,0.145976,0.266219,0.013217,0.000355


In [1002]:
# applying logistic regression
logistic_model = LogisticRegression(max_iter=1000, random_state=42)

start_train = time.perf_counter()
logistic_model.fit(X_cls_train_processed, y_cls_train)
end_train = time.perf_counter()
logistic_train_time = end_train - start_train
print("Training time: ", logistic_train_time)

# make predictions
start_predict = time.perf_counter()
y_cls_pred = logistic_model.predict(X_cls_test_processed)
end_predict = time.perf_counter()
logistic_prediction_time = end_predict - start_predict
print("Prediction time: ", logistic_prediction_time)

Training time:  0.014050600002519786
Prediction time:  0.0009740000241436064


In [1003]:
# accuracy
logistic_accuracy = accuracy_score(y_cls_test, y_cls_pred)
print("Accuracy: ", logistic_accuracy)

# precision
logistic_precision = precision_score(y_cls_test, y_cls_pred, zero_division=0)
print("Precision: ", logistic_precision)

# recall
logistic_recall = recall_score(y_cls_test, y_cls_pred, zero_division=0)
print("Recall: ", logistic_recall)

# F1 score
logistic_f1 = f1_score(y_cls_test, y_cls_pred, zero_division=0)
print("F1-score: ", logistic_f1)

logistic_results = {
    "Model": "Scikit-learn Logistic Regression",
    "Accuracy": logistic_accuracy,
    "Precision": logistic_precision,
    "Recall": logistic_recall,
    "F1": logistic_f1,
    "Training Time (s)": logistic_train_time,
    "Prediction Time (s)": logistic_prediction_time
}

logistic_results_df = pd.DataFrame([logistic_results])
display(logistic_results_df)

Accuracy:  0.7208333333333333
Precision:  0.7647058823529411
Recall:  0.8914285714285715
F1-score:  0.8232189973614775


,Model,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Scikit-learn Logistic Regression,0.720833,0.764706,0.891429,0.823219,0.014051,0.000974


# PART-B

In [1004]:
import pandas as pd
import numpy as np

df_manual = pd.read_csv("garments_worker_productivity.csv")
print("Shape:", df_manual.shape)
display(df_manual.head())

Shape: (1197, 15)


,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


In [1005]:
df_manual["MeetsTarget"] = (
    df_manual["actual_productivity"] >=
    df_manual["targeted_productivity"]
).astype(int)

# define feature and target
X_manual = df_manual.drop(
    columns=["actual_productivity", "MeetsTarget"]).copy()
y_manual_reg = df_manual["actual_productivity"].copy()
y_manual_cls = df_manual["MeetsTarget"].copy()

# train-test-split
X_manual_train = X_manual.iloc[train_indices].copy()
X_manual_test = X_manual.iloc[test_indices].copy()

y_manual_reg_train = y_manual_reg.iloc[train_indices].copy()
y_manual_reg_test = y_manual_reg.iloc[test_indices].copy()

y_manual_cls_train = y_manual_cls.iloc[train_indices].copy()
y_manual_cls_test = y_manual_cls.iloc[test_indices].copy()

In [1006]:
# identify numerical and categorical columns
numeric_features_manual = X_manual_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features_manual = X_manual_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features: \n",numeric_features_manual)
print("\nCategorical features: \n",categorical_features_manual)

Numerical features: 
 ['team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']

Categorical features: 
 ['date', 'quarter', 'department', 'day']


In [1007]:
# numeric missing values handling
numeric_medians = X_manual_train[numeric_features_manual].median()

print("Training medians: \n", numeric_medians)

# apply the median
X_manual_train[numeric_features_manual] = (
    X_manual_train[numeric_features_manual].fillna(numeric_medians) )

X_manual_test[numeric_features_manual] = (
    X_manual_test[numeric_features_manual].fillna(numeric_medians) )

# categorical missing values handling
categorical_modes = {}

for col in categorical_features_manual:
    categorical_modes[col] = (X_manual_train[col].mode()[0] )

print("Categorical modes:")
print(categorical_modes)

# fill missing values
for col in categorical_features_manual:
    X_manual_train[col] = (
        X_manual_train[col]
        .fillna(categorical_modes[col])
    )

    X_manual_test[col] = (
        X_manual_test[col]
        .fillna(categorical_modes[col])
    )

Training medians: 
 team                        6.00
targeted_productivity       0.75
smv                        15.26
wip                      1038.00
over_time                4080.00
incentive                   0.00
idle_time                   0.00
idle_men                    0.00
no_of_style_change          0.00
no_of_workers              34.00
dtype: float64
Categorical modes:
{'date': '1/11/2015', 'quarter': 'Quarter1', 'department': 'sweing', 'day': 'Sunday'}


In [1008]:
# capture the categories
train_categories = {}

for col in categorical_features_manual:
    train_categories[col] = (
        X_manual_train[col]
        .astype(str)
        .unique()
        .tolist()
    )

print(train_categories)

# convert to dummy variables
X_manual_train_encoded = pd.get_dummies(
    X_manual_train,
    columns=categorical_features_manual,
    dtype=int
)

X_manual_test_encoded = pd.get_dummies(
    X_manual_test,
    columns=categorical_features_manual,
    dtype=int
)

X_manual_test_encoded = X_manual_test_encoded.reindex(
    columns=X_manual_train_encoded.columns,
    fill_value=0
)

print("Train columns:", X_manual_train_encoded.shape[1])
print("Test columns:", X_manual_test_encoded.shape[1])

{'date': ['3/8/2015', '3/1/2015', '1/27/2015', '3/10/2015', '1/6/2015', '1/21/2015', '2/15/2015', '3/2/2015', '1/24/2015', '1/10/2015', '1/8/2015', '2/5/2015', '1/5/2015', '2/16/2015', '2/26/2015', '3/4/2015', '1/4/2015', '1/22/2015', '2/2/2015', '1/7/2015', '1/28/2015', '3/11/2015', '2/7/2015', '1/14/2015', '1/20/2015', '1/11/2015', '2/12/2015', '1/1/2015', '1/17/2015', '3/9/2015', '2/10/2015', '2/3/2015', '1/31/2015', '2/1/2015', '2/25/2015', '1/26/2015', '3/7/2015', '1/12/2015', '3/5/2015', '2/18/2015', '1/25/2015', '2/17/2015', '2/11/2015', '1/18/2015', '2/4/2015', '2/9/2015', '2/22/2015', '1/3/2015', '2/23/2015', '1/13/2015', '2/14/2015', '1/15/2015', '2/8/2015', '2/19/2015', '3/3/2015', '2/24/2015', '1/29/2015', '2/28/2015', '1/19/2015'], 'quarter': ['Quarter2', 'Quarter1', 'Quarter4', 'Quarter3', 'Quarter5'], 'department': ['finishing', 'finishing ', 'sweing'], 'day': ['Sunday', 'Tuesday', 'Wednesday', 'Monday', 'Saturday', 'Thursday']}
Train columns: 83
Test columns: 83


In [1009]:
# standardization
feature_means = X_manual_train_encoded.mean()
feature_stds = X_manual_train_encoded.std(ddof=0)

# apply scaling
X_manual_train_scaled = (
    X_manual_train_encoded - feature_means
) / feature_stds

X_manual_test_scaled = (
    X_manual_test_encoded - feature_means
) / feature_stds

# convert everything into numpy array
X_manual_train_final = X_manual_train_scaled.to_numpy(dtype=float)
X_manual_test_final = X_manual_test_scaled.to_numpy(dtype=float)
y_manual_reg_train_np = y_manual_reg_train.to_numpy(dtype=float)
y_manual_reg_test_np = y_manual_reg_test.to_numpy(dtype=float)
y_manual_cls_train_np = y_manual_cls_train.to_numpy(dtype=float)
y_manual_cls_test_np = y_manual_cls_test.to_numpy(dtype=float)


In [1010]:
# linear regression
# adding intercept column
X_train_lr = np.column_stack([
    np.ones(X_manual_train_final.shape[0]),
    X_manual_train_final
])

X_test_lr = np.column_stack([
    np.ones(X_manual_test_final.shape[0]),
    X_manual_test_final
])

In [1011]:
start_train = time.perf_counter()
A = X_train_lr.T @ X_train_lr
b = X_train_lr.T @ y_manual_reg_train_np
beta_manual = np.linalg.solve(A, b)
end_train = time.perf_counter()
manual_linear_train_time = end_train - start_train

# inspect the coefficients
print("Number of coefficients:", len(beta_manual))
print("\nIntercept:", beta_manual[0])

print(beta_manual)

Number of coefficients: 84

Intercept: 0.7324859741619647
[ 7.32485974e-01 -2.89366569e-02  6.91235501e-02 -7.64750928e-02
  1.97755595e-02 -5.28306687e-04  1.45847627e-02  1.82221969e-03
 -2.53956442e-02 -1.32661325e-02  9.02499406e-02  1.22416182e+00
 -6.22908340e-01  1.67303519e+00  2.47276966e+00  2.95926618e-01
 -3.13801661e+00 -1.29070212e+00 -2.31693908e+00 -1.01872081e-01
  7.15769834e-01 -1.03876839e+00 -4.74590850e+00 -7.63961496e-02
 -1.14818889e+00  1.09512633e+00  1.74208990e+00 -8.81057443e-02
 -3.65481849e+00  9.11926049e-01  2.87263673e-01 -7.58905665e-02
  2.35900206e+00  3.32114168e+00  1.24548231e+00 -2.15504360e+00
  3.55353418e-01  2.28871392e+00  3.12881182e-01 -3.03400078e+00
  3.07328732e-01 -5.87501643e-01 -9.22038953e-02  6.39638948e-01
 -1.20518481e+00 -4.60549882e+00 -1.10288427e+00  3.20138829e+00
  1.12098048e+00  1.73261457e+00 -8.66942090e-02 -3.44337962e+00
 -7.08602570e-02 -1.09402632e+00  1.20879659e+00 -2.07582321e+00
  1.24075137e+00  2.74809873e-01

In [1012]:
# make predictions
start_predict = time.perf_counter()
y_manual_reg_pred = X_test_lr @ beta_manual
end_predict = time.perf_counter()
manual_linear_prediction_time = end_predict - start_predict

# check predictions
prediction_check = pd.DataFrame({
    "Actual": y_manual_reg_test_np[:10],
    "Predicted": y_manual_reg_pred[:10]
})

display(prediction_check)

,Actual,Predicted
0,0.601944,0.838452
1,0.750621,0.713420
2,0.649662,0.785615
3,0.579511,0.620733
4,0.586465,0.740188
5,0.800129,0.743189
6,0.504687,0.630228
7,0.961785,0.820528
8,0.874028,0.759231
9,0.785864,0.797796


In [1013]:
# linear regression metrices
errors = y_manual_reg_test_np - y_manual_reg_pred

manual_linear_mae = np.mean(np.abs(errors)) # MAE
manual_linear_mse = np.mean(errors ** 2) # MSE
manual_linear_rmse = np.sqrt(manual_linear_mse) # RMSE

# R²
sse = np.sum(errors ** 2)
sst = np.sum(
    (y_manual_reg_test_np - np.mean(y_manual_reg_test_np)) ** 2
)

manual_linear_r2 = 1 - (sse / sst)

print("Manual Linear Regression Metrics")
print("MAE :", manual_linear_mae)
print("MSE :", manual_linear_mse)
print("RMSE:", manual_linear_rmse)
print("R²  :", manual_linear_r2)

# result table
manual_linear_results = pd.DataFrame([{
    "Model": "Manual Linear Regression",
    "MAE": manual_linear_mae,
    "RMSE": manual_linear_rmse,
    "R²": manual_linear_r2,
    "Training Time (s)": manual_linear_train_time,
    "Prediction Time (s)": manual_linear_prediction_time
}])

# compare with scikit learn
linear_comparison = pd.concat(
    [sklearn_regression_results, manual_linear_results], ignore_index=True)

display(linear_comparison)

Manual Linear Regression Metrics
MAE : 0.10896790339255613
MSE : 0.021308929593345724
RMSE: 0.1459757842703567
R²  : 0.2662191460358845


,Model,MAE,RMSE,R²,Training Time (s),Prediction Time (s)
0,Linear Regression,0.108968,0.145976,0.266219,0.01139,0.000493
1,Manual Linear Regression,0.108968,0.145976,0.266219,0.00122,0.000124


In [1014]:
# logistic regression

# Add intercept column
X_train_log = np.column_stack([
    np.ones(X_manual_train_final.shape[0]),
    X_manual_train_final
])

X_test_log = np.column_stack([
    np.ones(X_manual_test_final.shape[0]),
    X_manual_test_final
])

# sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [1015]:
# MANUAL LOGISTIC REGRESSION

n_samples = X_train_log.shape[0]
n_features = X_train_log.shape[1]

# Initialize coefficients with zeros
beta_logistic = np.zeros(n_features)

# Hyperparameters
learning_rate = 0.01
epochs = 10000

loss_history = []
start_train = time.perf_counter()

for epoch in range(epochs):

    # Linear combination
    z = X_train_log @ beta_logistic

    # Convert log-odds to probability
    p = sigmoid(z)

    # Calculate binary cross-entropy loss
    p_clipped = np.clip(p, 1e-15, 1 - 1e-15)
    loss = -np.mean(
        y_manual_cls_train_np * np.log(p_clipped)
        + (1 - y_manual_cls_train_np) * np.log(1 - p_clipped) )

    loss_history.append(loss)

    # Calculate gradient
    gradient = (
        X_train_log.T @ (p - y_manual_cls_train_np)
    ) / n_samples

    # Update coefficients
    beta_logistic -= learning_rate * gradient

end_train = time.perf_counter()
manual_logistic_train_time = end_train - start_train

print("Training completed.")
print("Final loss:", loss_history[-1])
print("Training time:", manual_logistic_train_time)
print("Number of coefficients:", len(beta_logistic))
print("Intercept:", beta_logistic[0])

Training completed.
Final loss: 0.45747406116879397
Training time: 1.0493560999748297
Number of coefficients: 84
Intercept: 1.3441380045639464


In [1016]:
start_predict = time.perf_counter()

# Calculate log-odds / linear score
z_test = X_test_log @ beta_logistic

# Convert log-odds into probabilities
y_manual_cls_probability = sigmoid(z_test)

# Convert probabilities into class predictions
y_manual_cls_pred = (
    y_manual_cls_probability >= 0.5
).astype(int)

end_predict = time.perf_counter()
manual_logistic_prediction_time = end_predict - start_predict
print("Prediction completed.")
print("Prediction time:", manual_logistic_prediction_time)

prediction_preview = pd.DataFrame({
    "Actual": y_manual_cls_test_np[:10].astype(int),
    "Probability_Class_1": y_manual_cls_probability[:10],
    "Predicted": y_manual_cls_pred[:10]
})

display(prediction_preview)

Prediction completed.
Prediction time: 0.00047879997873678803


,Actual,Probability_Class_1,Predicted
0,0,0.453138,0
1,1,0.792692,1
2,0,0.777213,1
3,0,0.348476,0
4,0,0.870041,1
5,1,0.865204,1
6,0,0.400518,0
7,1,0.833154,1
8,1,0.640907,1
9,0,0.271046,0


In [1017]:
# Convert to integer arrays
actual = y_manual_cls_test_np.astype(int)
predicted = y_manual_cls_pred.astype(int)

# Confusion matrix components
TP = np.sum((actual == 1) & (predicted == 1))
TN = np.sum((actual == 0) & (predicted == 0))
FP = np.sum((actual == 0) & (predicted == 1))
FN = np.sum((actual == 1) & (predicted == 0))

print("True Positives :", TP)
print("True Negatives :", TN)
print("False Positives:", FP)
print("False Negatives:", FN)

# Accuracy
manual_accuracy = (TP + TN) / (TP + TN + FP + FN)

# Precision
if TP + FP == 0:
    manual_precision = 0
else:
    manual_precision = TP / (TP + FP)

# Recall
if TP + FN == 0:
    manual_recall = 0
else:
    manual_recall = TP / (TP + FN)

# F1 Score
if manual_precision + manual_recall == 0:
    manual_f1 = 0
else:
    manual_f1 = (
        2 * manual_precision * manual_recall
        / (manual_precision + manual_recall)
    )

print("Accuracy :", manual_accuracy)
print("Precision:", manual_precision)
print("Recall   :", manual_recall)
print("F1 Score :", manual_f1)

# result table
manual_logistic_results = pd.DataFrame([{
    "Model": "Manual Logistic Regression",
    "Accuracy": manual_accuracy,
    "Precision": manual_precision,
    "Recall": manual_recall,
    "F1": manual_f1,
    "Training Time (s)": manual_logistic_train_time,
    "Prediction Time (s)": manual_logistic_prediction_time
}])

display(manual_logistic_results)

# comparision for classification
classification_comparison = pd.concat(
    [sklearn_classification_results, manual_logistic_results],
    ignore_index=True
)

display(classification_comparison)

True Positives : 157
True Negatives : 22
False Positives: 43
False Negatives: 18
Accuracy : 0.7458333333333333
Precision: 0.785
Recall   : 0.8971428571428571
F1 Score : 0.8373333333333334


,Model,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Manual Logistic Regression,0.745833,0.785,0.897143,0.837333,1.049356,0.000479


,Model,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Logistic Regression,0.720833,0.764706,0.891429,0.823219,0.021210,0.000498
1,Manual Logistic Regression,0.745833,0.785000,0.897143,0.837333,1.049356,0.000479


# PART-C

In [1018]:
# IMPROVED MANUAL LOGISTIC REGRESSION
beta_logistic_opt = np.zeros(X_train_log.shape[1])
learning_rate = 0.01
max_epochs = 10000
tolerance = 1e-6
loss_history_opt = []
start_train = time.perf_counter()

for epoch in range(max_epochs):
    # 1. Linear score
    z = X_train_log @ beta_logistic_opt

    # 2. Probability using sigmoid
    p = sigmoid(z)

    # 3. Calculate loss
    p_clipped = np.clip(p, 1e-15, 1 - 1e-15)
    loss = -np.mean(
        y_manual_cls_train_np * np.log(p_clipped)
        + (1 - y_manual_cls_train_np)
        * np.log(1 - p_clipped)
    )

    loss_history_opt.append(loss)

    # 4. Calculate gradient
    gradient = (
        X_train_log.T
        @ (p - y_manual_cls_train_np)
    ) / n_samples

    # 5. Check convergence
    if np.linalg.norm(gradient) < tolerance:
        print("Converged at epoch:", epoch + 1)
        break

    # 6. Update coefficients
    beta_logistic_opt -= learning_rate * gradient

end_train = time.perf_counter()
manual_logistic_opt_train_time = end_train - start_train

print("Training time:", manual_logistic_opt_train_time)
print("Iterations used:", len(loss_history_opt))
print("Initial loss:", loss_history_opt[0])
print("Final loss:", loss_history_opt[-1])
print("Loss reduction:", loss_history_opt[0] - loss_history_opt[-1])

Training time: 1.2057696999982
Iterations used: 10000
Initial loss: 0.6931471805599452
Final loss: 0.45747406116879397
Loss reduction: 0.2356731193911512


In [1019]:
# Prediction using optimized coefficients
start_predict = time.perf_counter()

# Linear score / log-odds
z_test_opt = X_test_log @ beta_logistic_opt

# Probability
y_manual_cls_probability_opt = sigmoid(z_test_opt)

# Class prediction
y_manual_cls_pred_opt = (
    y_manual_cls_probability_opt >= 0.5
).astype(int)

end_predict = time.perf_counter()
manual_logistic_opt_prediction_time = (end_predict - start_predict)
print("Prediction time:", manual_logistic_opt_prediction_time)

# check the predictions:
optimized_prediction_preview = pd.DataFrame({
    "Actual": y_manual_cls_test_np[:10].astype(int),
    "Probability_Class_1": y_manual_cls_probability_opt[:10],
    "Predicted": y_manual_cls_pred_opt[:10]
})

display(optimized_prediction_preview)

Prediction time: 0.0002500000409781933


,Actual,Probability_Class_1,Predicted
0,0,0.453138,0
1,1,0.792692,1
2,0,0.777213,1
3,0,0.348476,0
4,0,0.870041,1
5,1,0.865204,1
6,0,0.400518,0
7,1,0.833154,1
8,1,0.640907,1
9,0,0.271046,0


In [1020]:
actual = y_manual_cls_test_np.astype(int)
predicted_opt = y_manual_cls_pred_opt.astype(int)
TP_opt = np.sum((actual == 1) & (predicted_opt == 1))
TN_opt = np.sum((actual == 0) & (predicted_opt == 0))
FP_opt = np.sum((actual == 0) & (predicted_opt == 1))
FN_opt = np.sum((actual == 1) & (predicted_opt == 0))

accuracy_opt = ((TP_opt + TN_opt)/ (TP_opt + TN_opt + FP_opt + FN_opt) )
precision_opt = (TP_opt / (TP_opt + FP_opt)
    if (TP_opt + FP_opt) != 0
    else 0)

recall_opt = (TP_opt / (TP_opt + FN_opt)
    if (TP_opt + FN_opt) != 0
    else 0)

f1_opt = (2 * precision_opt * recall_opt/ (precision_opt + recall_opt)
    if (precision_opt + recall_opt) != 0
    else 0)

print("Optimized Manual Logistic Regression")
print("Accuracy :", accuracy_opt)
print("Precision:", precision_opt)
print("Recall   :", recall_opt)
print("F1       :", f1_opt)

Optimized Manual Logistic Regression
Accuracy : 0.7458333333333333
Precision: 0.785
Recall   : 0.8971428571428571
F1       : 0.8373333333333334


In [1021]:
# MANUAL LOGISTIC REGRESSION WITH L2 REGULARIZATION
beta_logistic_reg = np.zeros(X_train_log.shape[1])
learning_rate = 0.01
max_epochs = 15000
tolerance = 1e-6

# Regularization strength
lambda_reg = 0.01
loss_history_reg = []
start_train = time.perf_counter()

for epoch in range(max_epochs):
    # 1. Linear score
    z = X_train_log @ beta_logistic_reg

    # 2. Probability
    p = sigmoid(z)

    # 3. Avoid log(0)
    p_clipped = np.clip(p, 1e-15, 1 - 1e-15)

    # 4. Binary cross-entropy
    cross_entropy = -np.mean(
        y_manual_cls_train_np * np.log(p_clipped)
        + (1 - y_manual_cls_train_np)
        * np.log(1 - p_clipped)
    )

    # 5. L2 regularization
    # Do not penalize intercept beta[0]
    regularization = (
        lambda_reg / (2 * n_samples)
    ) * np.sum(beta_logistic_reg[1:] ** 2)

    # 6. Total loss
    loss = cross_entropy + regularization

    loss_history_reg.append(loss)

    # 7. Gradient of cross-entropy
    gradient = (
        X_train_log.T
        @ (p - y_manual_cls_train_np)
    ) / n_samples

    # 8. Add L2 gradient
    gradient[1:] += (
        lambda_reg / n_samples
    ) * beta_logistic_reg[1:]

    # 9. Check convergence
    if np.linalg.norm(gradient) < tolerance:
        print("Converged at epoch:", epoch + 1)
        break

    # 10. Update coefficients
    beta_logistic_reg -= learning_rate * gradient

end_train = time.perf_counter()
manual_logistic_reg_train_time = (end_train - start_train)

print("Final loss:", loss_history_reg[-1])
print("Training time:", manual_logistic_reg_train_time)
print("Iterations:", len(loss_history_reg))

Final loss: 0.45636951065155096
Training time: 2.4669701000093482
Iterations: 15000


In [1022]:
# Prediction using L2-regularized coefficients
start_predict = time.perf_counter()

# Calculate log-odds
z_test_reg = X_test_log @ beta_logistic_reg

# Convert log-odds to probability
y_manual_cls_probability_reg = sigmoid(z_test_reg)

# Convert probability to class
y_manual_cls_pred_reg = (
    y_manual_cls_probability_reg >= 0.5
).astype(int)

end_predict = time.perf_counter()

manual_logistic_reg_prediction_time = (
    end_predict - start_predict
)

print("Prediction time:",
      manual_logistic_reg_prediction_time)

# check few predictions
regularized_prediction_preview = pd.DataFrame({
    "Actual": y_manual_cls_test_np[:10].astype(int),
    "Probability_Class_1": y_manual_cls_probability_reg[:10],
    "Predicted": y_manual_cls_pred_reg[:10]
})

display(regularized_prediction_preview)

Prediction time: 0.00037460000021383166


,Actual,Probability_Class_1,Predicted
0,0,0.472564,0
1,1,0.789028,1
2,0,0.778367,1
3,0,0.340305,0
4,0,0.877530,1
5,1,0.873691,1
6,0,0.396397,0
7,1,0.838363,1
8,1,0.639836,1
9,0,0.280317,0


In [1023]:
# calculate everything manually
actual = y_manual_cls_test_np.astype(int)
predicted_reg = y_manual_cls_pred_reg.astype(int)

TP_reg = np.sum((actual == 1) & (predicted_reg == 1) )
TN_reg = np.sum((actual == 0) & (predicted_reg == 0) )
FP_reg = np.sum((actual == 0) & (predicted_reg == 1) )
FN_reg = np.sum((actual == 1) & (predicted_reg == 0) )

# calculate metrices
# Accuracy
accuracy_reg = (
    (TP_reg + TN_reg)
    / (TP_reg + TN_reg + FP_reg + FN_reg) )

# Precision
precision_reg = (
    TP_reg / (TP_reg + FP_reg)
    if (TP_reg + FP_reg) != 0
    else 0)

# Recall
recall_reg = (
    TP_reg / (TP_reg + FN_reg)
    if (TP_reg + FN_reg) != 0
    else 0)

# F1 Score
f1_reg = (
    2 * precision_reg * recall_reg
    / (precision_reg + recall_reg)
    if (precision_reg + recall_reg) != 0
    else 0)

print("Regularized Manual Logistic Regression")
print("Accuracy :", accuracy_reg)
print("Precision:", precision_reg)
print("Recall   :", recall_reg)
print("F1       :", f1_reg)

# final optimized result
manual_logistic_reg_results = pd.DataFrame([{
    "Model": "Manual Logistic Regression (L2)",
    "Accuracy": accuracy_reg,
    "Precision": precision_reg,
    "Recall": recall_reg,
    "F1": f1_reg,
    "Training Time (s)": manual_logistic_reg_train_time,
    "Prediction Time (s)": manual_logistic_reg_prediction_time
}])

display(manual_logistic_reg_results)

Regularized Manual Logistic Regression
Accuracy : 0.7333333333333333
Precision: 0.7761194029850746
Recall   : 0.8914285714285715
F1       : 0.8297872340425533


,Model,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Manual Logistic Regression (L2),0.733333,0.776119,0.891429,0.829787,2.46697,0.000375


In [1024]:
# compare logistic regression
logistic_comparison = pd.concat(
    [sklearn_classification_results,
    manual_logistic_results,
    manual_logistic_reg_results],
    ignore_index=True
)

display(logistic_comparison)

# compare manual models
manual_logistic_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    
    "Original Manual": [
        manual_accuracy,
        manual_precision,
        manual_recall,
        manual_f1
    ],
    
    "Manual + L2": [
        accuracy_reg,
        precision_reg,
        recall_reg,
        f1_reg
    ]
})

manual_logistic_comparison["Absolute Difference"] = (
    np.abs(
        manual_logistic_comparison["Manual + L2"]
        - manual_logistic_comparison["Original Manual"]
    ) )

display(manual_logistic_comparison)

,Model,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Logistic Regression,0.720833,0.764706,0.891429,0.823219,0.021210,0.000498
1,Manual Logistic Regression,0.745833,0.785000,0.897143,0.837333,1.049356,0.000479
2,Manual Logistic Regression (L2),0.733333,0.776119,0.891429,0.829787,2.466970,0.000375


,Metric,Original Manual,Manual + L2,Absolute Difference
0,Accuracy,0.745833,0.733333,0.012500
1,Precision,0.785000,0.776119,0.008881
2,Recall,0.897143,0.891429,0.005714
3,F1,0.837333,0.829787,0.007546


In [1025]:
learning_rates = [0.001, 0.01, 0.05, 0.1]
learning_rate_results = []

for lr in learning_rates:

    beta_test = np.zeros(X_train_log.shape[1])
    start = time.perf_counter()

    for epoch in range(50000):

        z = X_train_log @ beta_test
        p = sigmoid(z)

        p_clipped = np.clip(p, 1e-15, 1 - 1e-15)

        gradient = (
            X_train_log.T
            @ (p - y_manual_cls_train_np)
        ) / n_samples

        # L2 regularization
        gradient[1:] += (
            lambda_reg / n_samples
        ) * beta_test[1:]

        if np.linalg.norm(gradient) < tolerance:
            break

        beta_test -= lr * gradient

    end = time.perf_counter()

    # Test predictions
    test_probability = sigmoid(
        X_test_log @ beta_test
    )

    test_prediction = (
        test_probability >= 0.5
    ).astype(int)

    # -----------------------------
    # Classification metrics
    # -----------------------------

    actual = y_manual_cls_test_np.astype(int)
    predicted = test_prediction.astype(int)

    TP = np.sum((actual == 1) & (predicted == 1))
    TN = np.sum((actual == 0) & (predicted == 0))
    FP = np.sum((actual == 0) & (predicted == 1))
    FN = np.sum((actual == 1) & (predicted == 0))

    accuracy = (
        (TP + TN)
        / (TP + TN + FP + FN)
    )

    precision = (
        TP / (TP + FP)
        if (TP + FP) != 0 else 0
    )

    recall = (
        TP / (TP + FN)
        if (TP + FN) != 0 else 0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if (precision + recall) != 0 else 0
    )

    # Store results
    learning_rate_results.append({
        "Learning Rate": lr,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Iterations": epoch + 1,
        "Training Time (s)": end - start
    })


learning_rate_results_df = pd.DataFrame(
    learning_rate_results
)

display(learning_rate_results_df)

,Learning Rate,Accuracy,Precision,Recall,F1,Iterations,Training Time (s)
0,0.001,0.745833,0.782178,0.902857,0.838196,50000,4.533952
1,0.010,0.725000,0.773869,0.880000,0.823529,50000,5.021912
2,0.050,0.725000,0.773869,0.880000,0.823529,50000,4.677381
3,0.100,0.725000,0.773869,0.880000,0.823529,50000,4.640201


In [1026]:
lambda_values = [0, 0.001, 0.01, 0.1, 1.0]
lambda_results = []

for lambda_value in lambda_values:

    beta_test = np.zeros(X_train_log.shape[1])

    start = time.perf_counter()

    for epoch in range(10000):

        # 1. Linear score
        z = X_train_log @ beta_test

        # 2. Probability
        p = sigmoid(z)

        # 3. Gradient of cross-entropy
        gradient = (
            X_train_log.T
            @ (p - y_manual_cls_train_np)
        ) / n_samples

        # 4. Add L2 regularization
        if lambda_value != 0:
            gradient[1:] += (
                lambda_value / n_samples
            ) * beta_test[1:]

        # 5. Check convergence
        if np.linalg.norm(gradient) < tolerance:
            break

        # 6. Update coefficients
        beta_test -= learning_rate * gradient

    end = time.perf_counter()

    # Test predictions
    test_probability = sigmoid(X_test_log @ beta_test)
    test_prediction = (
        test_probability >= 0.5
    ).astype(int)

    # -----------------------------
    # Classification metrics
    # -----------------------------

    actual = y_manual_cls_test_np.astype(int)
    predicted = test_prediction.astype(int)

    TP = np.sum((actual == 1) & (predicted == 1))
    TN = np.sum((actual == 0) & (predicted == 0))
    FP = np.sum((actual == 0) & (predicted == 1))
    FN = np.sum((actual == 1) & (predicted == 0))

    # Accuracy
    accuracy = (
        (TP + TN)
        / (TP + TN + FP + FN)
    )

    # Precision
    precision = (
        TP / (TP + FP)
        if (TP + FP) != 0
        else 0
    )

    # Recall
    recall = (
        TP / (TP + FN)
        if (TP + FN) != 0
        else 0
    )

    # F1 Score
    f1 = (
        2 * precision * recall
        / (precision + recall)
        if (precision + recall) != 0
        else 0
    )

    # Store results
    lambda_results.append({
        "Lambda": lambda_value,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Iterations": epoch + 1,
        "Training Time (s)": end - start
    })

lambda_results_df = pd.DataFrame(lambda_results)

display(lambda_results_df)

,Lambda,Accuracy,Precision,Recall,F1,Iterations,Training Time (s)
0,0.000,0.745833,0.785,0.897143,0.837333,10000,0.777695
1,0.001,0.745833,0.785,0.897143,0.837333,10000,0.793172
2,0.010,0.745833,0.785,0.897143,0.837333,10000,0.760597
3,0.100,0.745833,0.785,0.897143,0.837333,10000,0.672178
4,1.000,0.745833,0.785,0.897143,0.837333,10000,0.965513


In [1027]:
# FINAL MANUAL LOGISTIC REGRESSION
final_learning_rate = 0.01
final_lambda = 0.001
final_max_epochs = 50000
final_tolerance = 1e-6
beta_final = np.zeros(X_train_log.shape[1])
final_loss_history = []
start_train = time.perf_counter()

for epoch in range(final_max_epochs):
    # Linear combination
    z = X_train_log @ beta_final

    # Predicted probabilities
    p = sigmoid(z)

    # Prevent log(0)
    p_clipped = np.clip(p, 1e-15, 1 - 1e-15)

    # Cross-entropy loss
    cross_entropy = -np.mean(
        y_manual_cls_train_np * np.log(p_clipped)
        + (1 - y_manual_cls_train_np)
        * np.log(1 - p_clipped))

    # L2 regularization
    regularization = (
        final_lambda / (2 * n_samples)
    ) * np.sum(beta_final[1:] ** 2)

    # Total loss
    loss = cross_entropy + regularization
    final_loss_history.append(loss)

    # Gradient
    gradient = (
        X_train_log.T
        @ (p - y_manual_cls_train_np)
    ) / n_samples

    # Do not regularize intercept
    gradient[1:] += (
        final_lambda / n_samples
    ) * beta_final[1:]

    # Convergence check
    if np.linalg.norm(gradient) < final_tolerance:
        print("Converged at epoch:", epoch + 1)
        break

    # Update coefficients
    beta_final -= final_learning_rate * gradient

end_train = time.perf_counter()
final_manual_logistic_train_time = (end_train - start_train)

print("Final Manual Logistic Regression")
print("Learning rate:", final_learning_rate)
print("Lambda:", final_lambda)
print("Final loss:", final_loss_history[-1])
print("Iterations:", len(final_loss_history))
print("Training time:", final_manual_logistic_train_time)

Final Manual Logistic Regression
Learning rate: 0.01
Lambda: 0.001
Final loss: 0.45543028159153687
Iterations: 50000
Training time: 6.547148900048342


In [1028]:
# FINAL CLASSIFICATION COMPARISON
final_classification_comparison = pd.DataFrame([
    {"Model": "Scikit-learn Logistic Regression",
     "Accuracy": logistic_accuracy,
     "Precision": logistic_precision,
     "Recall": logistic_recall,
     "F1": logistic_f1,
     "Training Time (s)": logistic_train_time,
     "Prediction Time (s)": logistic_prediction_time},
     
    {"Model": "Manual Logistic Regression (L2)",
     "Accuracy": accuracy_final,
     "Precision": precision_final,
     "Recall": recall_final,
     "F1": f1_final,
     "Training Time (s)": final_manual_logistic_train_time,
     "Prediction Time (s)": final_manual_logistic_prediction_time}
])

display(final_classification_comparison)

# DIFFERENCE BETWEEN SKLEARN AND MANUAL
classification_difference_final = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    "Scikit-learn": [logistic_accuracy, logistic_precision, logistic_recall, logistic_f1],
    "Manual L2": [accuracy_final, precision_final, recall_final, f1_final]
})

classification_difference_final["Absolute Difference"] = (
    np.abs(
        classification_difference_final["Scikit-learn"]
        - classification_difference_final["Manual L2"] ) )

display(classification_difference_final)

,Model,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Scikit-learn Logistic Regression,0.720833,0.764706,0.891429,0.823219,0.014051,0.000974
1,Manual Logistic Regression (L2),0.741667,0.780488,0.903955,0.837696,6.547149,0.000341


,Metric,Scikit-learn,Manual L2,Absolute Difference
0,Accuracy,0.720833,0.741667,0.020833
1,Precision,0.764706,0.780488,0.015782
2,Recall,0.891429,0.903955,0.012526
3,F1,0.823219,0.837696,0.014477


In [1029]:
# FINAL REGRESSION COMPARISON

final_regression_comparison = pd.DataFrame([
    {
        "Model": "Scikit-learn Linear Regression",
        "MAE": linear_mae,
        "RMSE": linear_rmse,
        "R²": linear_r2,
        "Training Time (s)": linear_train_time,
        "Prediction Time (s)": linear_prediction_time
    },
    {
        "Model": "Manual Linear Regression",
        "MAE": manual_linear_mae,
        "RMSE": manual_linear_rmse,
        "R²": manual_linear_r2,
        "Training Time (s)": manual_linear_train_time,
        "Prediction Time (s)": manual_linear_prediction_time
    }
])

display(final_regression_comparison)

# DIFFERENCE BETWEEN SKLEARN AND MANUAL REGRESSION

regression_difference_final = pd.DataFrame({
    "Metric": [
        "MAE",
        "RMSE",
        "R²"
    ],
    "Scikit-learn": [
        linear_mae,
        linear_rmse,
        linear_r2
    ],
    "Manual": [
        manual_linear_mae,
        manual_linear_rmse,
        manual_linear_r2
    ]
})

regression_difference_final["Absolute Difference"] = (
    np.abs(
        regression_difference_final["Scikit-learn"]
        - regression_difference_final["Manual"]
    )
)

display(regression_difference_final)

,Model,MAE,RMSE,R²,Training Time (s),Prediction Time (s)
0,Scikit-learn Linear Regression,0.108968,0.145976,0.266219,0.013217,0.000355
1,Manual Linear Regression,0.108968,0.145976,0.266219,0.001220,0.000124


,Metric,Scikit-learn,Manual,Absolute Difference
0,MAE,0.108968,0.108968,5.689893e-16
1,RMSE,0.145976,0.145976,7.494005e-16
2,R²,0.266219,0.266219,7.771561e-15
